In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error


from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    cross_val_score,
    train_test_split,
)

In [7]:
base = "./data/"


test = pd.read_csv(base + "test.csv")
train = pd.read_csv(base + "train.csv")

In [8]:
SEED = 24
target_column = "SalePrice"
np.random.seed(SEED)

real_data_test = test.copy()

test_size = 0.2
data_train, data_test, Y_train, Y_test = train_test_split(
    train[train.columns.drop(target_column)],
    np.array(train[target_column]),
    test_size=test_size,
    random_state=SEED,
)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1444,2905,20,NaN,125.0,31250,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2006,WD,Normal
1445,2906,90,RM,78.0,7020,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,11,2006,WD,Normal
1446,2907,160,RM,41.0,2665,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2006,WD,Normal
1447,2908,20,RL,58.0,10172,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,10,2006,WD,Normal


In [44]:
data_train.head()
data_train.info()
# Alley - empty, can delete
# mostly empty: Alley, MasVnrType, FireplaceQu, PoolQC, Fence, MiscFeature, 
# a bit empty: LotFrontage, BsmtQual, BsmtCond, BsmtExposure, BsmtFinType1, BsmtFinType2, GarageType, 
#              GarageYrBlt, GarageFinish, GarageQual, GarageCond





# this columnds can be converted to numeric (like a rating): BsmtQual, BsmtCond, BsmtExposure, BsmtFinType1, BsmtFinType2
# one hot for: GarageType, GarageFinish, GarageQual, GarageCond, FireplaceQu, Fence

# do smth with the dates (maybe group them by some period, ie decades): GarageYrBlt, YrSold, YearBuilt

# YearRemodAdd - here we can compare with build date, if they are different - can set new property like: has-remod

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 80 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [9]:
# check outliers 
numeric_df = data.select_dtypes(include=[np.number])

# Настройка подграфиков
n_cols = 2
n_rows = int(np.ceil(len(numeric_df.columns) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))

axes = axes.flatten()

for i, col in enumerate(numeric_df.columns):
    df = numeric_df[col].dropna().astype(float)

    x_min, x_max = df.min(), df.max()
    y_max = df.value_counts(bins=10).max()

    axes[i].hist(df, bins=20, color='skyblue', edgecolor='black')
    axes[i].set_title(f"Histogram of {col}")
    axes[i].set_xlim(x_min - 1, x_max + 1)
    axes[i].set_ylim(0, y_max + 5)
    axes[i].grid(True)

# Удалим пустые графики если столбцов меньше, чем подграфиков
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

NameError: name 'data' is not defined

In [10]:
def _get_first_mode_value(data_series: pd.Series) -> str| int:
    return data_series.mode().values[0]


def replace_nan_with_mode(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    data = data.fillna(data.mode().iloc[0])
    return data


def get_cleaned_data(data: pd.DataFrame):
    cleaned_data = data.copy()

    # FireplaceQu, Fence - set to nan value - mode
    cleaned_data[cleaned_data['FireplaceQu'].isnull()] = _get_first_mode_value(cleaned_data['FireplaceQu'])
    cleaned_data[cleaned_data['Fence'].isnull()] = _get_first_mode_value(cleaned_data['Fence'])

    # Set median value: LotFrontage
    cleaned_data[cleaned_data['LotFrontage'].isnull()] = cleaned_data['LotFrontage'].median()

    # Delete columns (maybe update later): MiscFeature, BsmtFinSF2 and BsmtUnfSF (ничего не показывает)
    data.drop(inplace=True, columns=["MiscFeature", "BsmtFinSF2", "BsmtUnfSF"])

    # Delete columns with mostly null values:
    data.drop(inplace=True, columns=["Alley", "PoolQC"])

    rating_media = cleaned_data.rating.median().astype(float).round(1)
    cleaned_data["rating"] = (
        cleaned_data["rating"].astype(float).round(1).fillna(rating_media)
    )

    return cleaned_data

In [11]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_boxplots(df, cols_per_row=3):
    numeric_cols = df.select_dtypes(include='number').columns
    num_cols = len(numeric_cols)
    num_rows = -(-num_cols // cols_per_row)  # округление вверх

    fig, axes = plt.subplots(num_rows, cols_per_row, figsize=(6 * cols_per_row, 4 * num_rows))
    axes = axes.flatten()

    for i, col in enumerate(numeric_cols):
        sns.boxplot(y=df[col], ax=axes[i], color='skyblue')
        axes[i].set_title(col)
        axes[i].grid(True)

    # Отключаем пустые сабплоты
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()


plot_boxplots(data)

NameError: name 'data' is not defined

In [12]:
def linear_regression(X_train, X_test, Y_train):
    linear_regressor = LinearRegression()
    linear_regressor.fit(X_train, Y_train)
    Y_pred = linear_regressor.predict(X_test)
    return Y_pred


def linear_ridge(X_train, X_test, Y_train):
    linear_ridge = Ridge()
    linear_ridge.fit(X_train, Y_train)
    Y_pred = linear_ridge.predict(X_test)
    return Y_pred

def split_feature_columns(
    data: pd.DataFrame, target_column: str | None = None
) -> tuple[list, list]:
    continuous_columns = [
        key for key in data.keys() if data[key].dtype in ("int64", "float64")
    ]
    categorical_columns = [key for key in data.keys() if data[key].dtype == "object"]

    if target_column:
        continuous_columns.remove(target_column)

    print(
        f"Continuous : {len(continuous_columns)}, Categorical : {len(categorical_columns)}"
    )
    return continuous_columns, categorical_columns

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.discriminant_analysis import StandardScaler


class BaseDataPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, needed_columns: list[str] | None = None):
        """
        :param needed_columns: if not None select these columns from the dataframe
        """
        self.scaler = StandardScaler()

        self.needed_columns: list[str] | None = needed_columns
        self._selected_columns: list[str] | set[str] | None = None

    def get_data(self, data: pd.DataFrame) -> pd.DataFrame:
        """
        Returns a dataframe with only needed columns
        :param data: pd.DataFrame with all data
        """
        if self.needed_columns is not None:
            return data[self.needed_columns]
        return data

    def fit(self, data, *args) -> "BaseDataPreprocessor":
        """
        Prepares the class for future transformations
        :param data: pd.DataFrame with all available columns
        :return: self
        """
        df = data.copy()
        df = self.get_data(df)

        df = df.select_dtypes(include=["int", "float"])
        self._selected_columns = df.columns.tolist()
        self.scaler.fit(df)

        return self

    def transform(self, data: pd.DataFrame) -> np.array:
        """
        Transforms features so that they can be fed into the regressors
        :param data: pd.DataFrame with all available columns
        :return: np.array with preprocessed features
        """
        df = data.copy()

        if self._selected_columns:
            df = df[self._selected_columns]

        return self.scaler.transform(df)


In [14]:
continuous_columns, categorical_columns = split_feature_columns(data_train)

X_train = data_train[continuous_columns]
X_test = data_test[continuous_columns]

X_train_rp = replace_nan_with_mode(X_train)
X_test_rp = replace_nan_with_mode(X_test)

Continuous : 37, Categorical : 43


In [15]:
ridge_pred = linear_ridge(X_train_rp, X_test_rp, Y_train)
ridge__mae = mean_absolute_error(Y_test, ridge_pred)
ridge__mse = mean_squared_error(Y_test, ridge_pred)
print(f"Ridge MAE : {ridge__mae}")
print(f"Ridge MSE : {ridge__mse}\n")

Ridge MAE : 22686.82010039725
Ridge MSE : 1501205317.7669666



In [ ]:
from sklearn.base import TransformerMixin, BaseEstimator
import pandas as pd
from sklearn.impute import SimpleImputer



class DataFrameSimpleImputer(BaseEstimator, TransformerMixin):
    def __init__(self, strategy="mean"):
        self.strategy = strategy
        self.imputer = SimpleImputer(strategy=self.strategy)
        self.columns = None

    def fit(self, X, y=None):
        self.columns = X.columns
        self.imputer.fit(X)
        return self

    def transform(self, X):
        X_imputed = self.imputer.transform(X)
        return pd.DataFrame(X_imputed, columns=self.columns, index=X.index)


In [60]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import root_mean_squared_error

quality_mapping = {
    "Ex": 5,
    "Gd": 4,
    "TA": 3,
    "Fa": 2,
    "Po": 1,
    "NA": 0,
}
quality_mapping_keys = list(quality_mapping.keys())

categorical_quality_columns = ["GarageCond", "GarageQual", "PoolQC", "FireplaceQu", "KitchenQual", "HeatingQC", "BsmtCond", "BsmtQual", "ExterCond", "ExterQual"]



def get_pipeline_preprocessors(continuous_columns, categorical_columns):
    numerical_pipeline = Pipeline(
        steps=[
            ("imputer", DataFrameSimpleImputer(strategy="median")),
            ("scaler", BaseDataPreprocessor(needed_columns=continuous_columns)),
        ]
    )

    # categorical_quality_pipeline = Pipeline(
    #     steps=[
    #         ("imputer", DataFrameSimpleImputer(strategy="most_frequent")),
    #         ("ordinal_encoder", OrdinalEncoder(
    #             categories=[
    #                 quality_mapping_keys
    #             ] * len(categorical_quality_columns),
    #             handle_unknown="use_encoded_value",
    #             unknown_value=-1,
    #         )),
    #     ]
    # )

    # other_categorical_features = list(set(categorical_columns) - set(categorical_quality_columns))
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", DataFrameSimpleImputer(strategy="most_frequent")),
            ("onehot_encoder", OneHotEncoder(handle_unknown='ignore')),
        ]
    )

    preprocessors = [
        ("numerical", numerical_pipeline, continuous_columns),
        ("categorical", categorical_pipeline, categorical_columns),
        # ("categorical", other_categorical_pipeline, other_categorical_features),
        # ("categorical_quality", categorical_quality_pipeline, categorical_quality_columns)
    ]

    return ColumnTransformer(
        transformers=preprocessors,
    )


def make_pipeline(**kwargs):
    preprocessors = get_pipeline_preprocessors(**kwargs)

    grid_param = {'alpha': range(1, 10)}

    rmse_scorer = make_scorer(
        root_mean_squared_error, greater_is_better=False
    )

    sgd_grid = GridSearchCV(
        estimator=Ridge(),
        param_grid=grid_param,
        scoring=rmse_scorer,
        refit=True,
        cv=KFold(n_splits=5, shuffle=True, random_state=42),
        n_jobs=-1,
    )

    pipe = Pipeline(
        steps=[
            ("preprocessor", preprocessors),
            ("sgd_grid", sgd_grid),
        ],
    )

    return pipe

In [ ]:
# 32431.35 - with all cat 
# 34349.08 - split cat by feature type

In [62]:
X_train = data_train[continuous_columns + categorical_columns]
X_test = data_test[continuous_columns + categorical_columns]

pipe = make_pipeline(continuous_columns=continuous_columns, categorical_columns=categorical_columns)
pipe.fit(X_train, Y_train)
pipe_pred = pipe.predict(X_test)

pipe_ridge__rmse = root_mean_squared_error(Y_test, pipe_pred)
print(f"Ridge RMSE : {pipe_ridge__rmse}\n")

# pipe_ridge__mae = mean_absolute_error(Y_test, pipe_pred)
# pipe_ridge__mse = mean_squared_error(Y_test, pipe_pred)

# print(f"Ridge MAE : {pipe_ridge__mae}")
# print(f"Ridge MSE : {pipe_ridge__mse}\n")

Ridge RMSE : 32431.356646600987



/Users/nikita/.local/share/virtualenvs/kaggle-SPn0DasX/lib/python3.12/site-packages/sklearn/pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [58]:
X_test[:-10]

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,GarageCond,GarageQual,PoolQC,FireplaceQu,KitchenQual,HeatingQC,BsmtCond,BsmtQual,ExterCond,ExterQual
0,1,60,65.0,8450,7,5,2003,2003,196.0,706,...,TA,TA,NaN,NaN,Gd,Ex,TA,Gd,TA,Gd
1,2,20,80.0,9600,6,8,1976,1976,0.0,978,...,TA,TA,NaN,TA,TA,Ex,TA,Gd,TA,TA
2,3,60,68.0,11250,7,5,2001,2002,162.0,486,...,TA,TA,NaN,TA,Gd,Ex,TA,Gd,TA,Gd
3,4,70,60.0,9550,7,5,1915,1970,0.0,216,...,TA,TA,NaN,Gd,Gd,Gd,Gd,TA,TA,TA
4,5,60,84.0,14260,8,5,2000,2000,350.0,655,...,TA,TA,NaN,TA,Gd,Ex,TA,Gd,TA,Gd
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1445,1446,85,70.0,8400,6,5,1966,1966,0.0,187,...,TA,TA,NaN,NaN,TA,Gd,TA,TA,TA,TA
1446,1447,20,NaN,26142,5,7,1962,1962,189.0,593,...,TA,TA,NaN,NaN,TA,TA,TA,TA,TA,TA
1447,1448,60,80.0,10000,8,5,1995,1996,438.0,1079,...,TA,TA,NaN,TA,Gd,Ex,TA,Gd,TA,Gd
1448,1449,50,70.0,11767,4,7,1910,2000,0.0,0,...,TA,Fa,NaN,NaN,TA,Gd,TA,Fa,TA,TA


In [63]:
pipe_real_pred = pipe.predict(real_data_test)

df = pd.DataFrame(pipe_real_pred, columns=["SalePrice"])

df.insert(0, "Id", real_data_test.Id.values)
df.to_csv("output.csv", index=False)

/Users/nikita/.local/share/virtualenvs/kaggle-SPn0DasX/lib/python3.12/site-packages/sklearn/pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
